In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from transformers import BertTokenizer
from sklearn.metrics import precision_score, accuracy_score, f1_score, balanced_accuracy_score

/opt/miniconda3/envs/splade_class/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def count_tokens(text):
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    
    # Initialize a frequency vector (size of BERT vocabulary)
    vocab_size = len(tokenizer.vocab)
    frequency_vector = np.zeros(vocab_size, dtype=int)
    
    # Use numpy's histogram to count occurrences of each token ID
    frequency_vector, _ = np.histogram(token_ids, bins=np.arange(vocab_size + 1))
    return frequency_vector

def get_coefficients(texts, labels):
    frequency_matrix = np.array([count_tokens(text) for text in texts])
    X_vectorized = np.array([count_tokens(text) for text in texts])
    # Train an SVM with a linear kernel
    svm = SVC(kernel='linear', random_state=42, verbose = True)
    svm.fit(X_vectorized, labels)
    coefficients = svm.coef_.flatten()

    # Test the classifier
    y_vectorised = np.array([count_tokens(text) for text in test_docs.text.tolist()])
    y_pred = svm.predict(y_vectorised)
    y_true = test_docs.sensitivity.tolist()
    
    precision = precision_score(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred) 
    bac = balanced_accuracy_score(y_true, y_pred)
    
    # Print the metrics
    print(f"Precision: {precision}")
    print(f"Accuracy: {accuracy}")
    print(f"F1 Score: {f1}")
    print(f"Balanced Accuracy: {bac}")
    return coefficients

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
training_docs = pd.read_pickle("/nfs/primary/sas_reranker/d2qmm_ohsumed_training_data.pkl")
test_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")

In [4]:
training_docs_sensitivity_0 = training_docs[training_docs['sensitivity'] == 0].sample(n=50, random_state=42)
training_docs_sensitivity_1 = training_docs[training_docs['sensitivity'] == 1].sample(n=50, random_state=42)

# Combine the selected rows
training_docs_selected = pd.concat([training_docs_sensitivity_0, training_docs_sensitivity_1])

In [ ]:
coefficients = get_coefficients(training_docs_selected.text.tolist(), training_docs_selected.sensitivity.tolist())

[LibSVM].*.*
optimization finished, #iter = 259
obj = -0.180973, rho = -0.009603
nSV = 96, nBSV = 0
Total nSV = 96


In [ ]:
# Get the token IDs (BERT's vocabulary)
tokens = tokenizer.convert_ids_to_tokens(np.arange(len(tokenizer.vocab)))

# Map coefficients to their corresponding tokens
feature_importance = pd.DataFrame({
    'Token': tokens,
    'Coefficient': coefficients
})

# Sort by absolute value of coefficients to get most important features
feature_importance_sorted = feature_importance.sort_values(by='Coefficient', ascending=False)

# Show the top 10 most important tokens
feature_importance_sorted.head(20)